# DA3METRIC-LARGE: KITTI Fine-Tuning & Evaluation Pipeline
## Final Recipe (Eigen Split) -- Ablation-Informed Loss Combinations

**Pipeline steps:**
1. Fine-tune DA3METRIC-LARGE on KITTI Eigen *train* with 90/10 train-val split
2. Resume from `latest.pth` (saved every epoch) or any periodic checkpoint
3. Evaluate best checkpoint on the KITTI Eigen *test* split (unscaled metrics only)
4. Per-epoch analysis using periodic checkpoints (val_rmse + test_rmse vs epoch)

---

### Loss combinations from ablation study

| Tag  | Composition                                                                   |
|------|-------------------------------------------------------------------------------|
| `A1` | 1.0*MAECanonical + 1.0*SILog + 0.1*GradientLoss                               |
| `A2` | 1.0*MAECanonical + 1.0*SILog                                                   |
| `A3` | 1.0*MAECanonical + 1.0*SILog + 0.1*GradientLoss + 0.5*BerHu                   |
| `A4` | 1.0*MAECanonical + 1.0*SILog + 0.1*GradientLoss + 0.5*MaskedDepth             |
| `B1` | 0.5*MAECanonical + 1.0*SILog + 1.0*BerHu + 0.5*L1 + 0.1*GradientLoss         |
| `B2` | 1.0*MAECanonical + 1.0*SILog + 1.0*BerHu + 0.5*L1 + 0.1*GradientLoss         |
| `C`  | 1.0*MAECanonical (SGD + Nesterov, LR = 1e-4)                                   |

> **MAECanonical scale fix:** MAECanonical operates in inverse-depth space (1/m).
> Clamping pred to [MAE_DEPTH_FLOOR, MAX_DEPTH] before inversion bounds
> 1/pred to [1/MAX_DEPTH, 1/MAE_DEPTH_FLOOR]. With default MAE_DEPTH_FLOOR=0.5 m,
> SILog (~0.04-0.5) so recipe weights actually balance the components.
> Previously clamp was at EPS=1e-6 giving 1/pred up to 1e6 which made
> mae_canonical dominate by 5 orders of magnitude.

### Fixed training recipe

| Knob                 | Value                                                                    |
|----------------------|--------------------------------------------------------------------------|
| Architecture         | `full` or `decoder_only` (encoder frozen)                                          |
| Augmentation         | `base` (horizontal flip + mild brightness/contrast jitter)               |
| Optimiser/Scheduler  | AdamW + cosine_warmup  (C: SGD + Nesterov)                               |
| LR                   | 1e-6 (from 2e-6)  (C: 1e-4)                                              |
| Weight decay         | 1e-4                                                                     |
| Epochs               | 50, patience=30 on val_rmse (pixel-weighted)                             |
| Effective batch      | 32 (BS 8 x Accum 4)                                                      |
| Warmup               | 1 epochs (from 2 epochs)                                                 |
| Grad clip            | max-norm 1.0                                                             |
| Mixed precision      | AMP fp16                                                                 |
| Checkpointing        | latest.pth (every epoch) + checkpoint_epoch_NNNN.pth (every 1, permanent)|
| Best model           | best_model.pth -- saved on val_rmse improvement                          |
| Train/val split      | 90/10 of TRAIN_ROOT (seed=42); test = TEST_ROOT                          |

### Evaluation policy

- **Unscaled metrics only** -- median/mean/affine alignment removed entirely.
- **Full-frame** (Eigen crop) always reported for every test frame.
- **Mask-category** rows additionally emitted when MASK_ROOT is set.
- **Pixel-weighted RMSE** = sqrt(sum_SSE / sum_N_pixels) across all rows in group.
- **Summary**: full-frame first, then super-category, then sub-category.


## 0 · Imports

In [1]:
import os
# Set the visible device to index 1 (GPU 1)
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
import csv, glob, json, logging, math, os, shutil, sys, time, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# Multiprocessing: must be set BEFORE any DataLoader is created.
# "spawn" avoids fork-safety issues that cause the
#   "can only test a child process" AssertionError between epochs.
import multiprocessing
try:
    multiprocessing.set_start_method("spawn", force=True)
except RuntimeError:
    pass  # already set (e.g. cell re-run)

import cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
# ─────────────────────────────────────────────────────────────────────────────


## 1 · Configuration

> ✏️ **Edit paths and hyperparameters here before running.**  
> All output directories are created automatically.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Paths
KITTI_ROOT       = "/home/arka/Downloads/MMDE/datasets/KITTI/complete"
TRAIN_ROOT       = os.path.join(KITTI_ROOT, "train")
TEST_ROOT        = os.path.join(KITTI_ROOT, "test")
CACHE_DIR        = "/home/arka/Downloads/MMDE/cache"
OUTPUT_BASE_ROOT = "/home/arka/Downloads/MMDE/outputs/fully_supervised_FT_output/da3_kitti_finetune_final_outputs_complete_v4_FULL_ARCH"

# Model
MODEL_ID         = "depth-anything/DA3METRIC-LARGE"
ORTHO_NORM       = 300.0
FOCAL_LENGTH_X   = 721.5377
USE_MODEL_FOCAL  = False

# ===== LOSS COMBINATION SELECTOR =============================================
# Choose one of: "A1","A2","A3","A4","B1","B2","C"
LOSS_NAME        = "A1"
# =============================================================================

ARCH_MODE        = "full"          # "decoder_only" or "full"
AUG_MODE         = "base"
OPTIM_MODE       = "adamw"
SCHED_MODE       = "cosine_warmup"

# Training
EPOCHS           = 50
BATCH_SIZE       = 4             # from 8 from 16 from 8 # from 4 # from 2
ACCUM_STEPS      = 2             # from 4 effective BS = 32 from 2 | effective batch = 16 (about 10 - 20+ gb vram (16*1 the vram hits 32 gb)) from 8 (about 6 - 12+ gb vram)
LEARNING_RATE    = 2e-7          # (for full 5e-7 to 1e-7 is recommended) from 1e-6 from 2e-6, 1.0e-5 is also too hot after warmup epochs # seems like 2e-5 is too hot after epoch 4, so fixed scheduler logic and reduced lr
WEIGHT_DECAY     = 1e-4
WARMUP_EPOCHS    = 1             # from 2
MAX_GRAD_NORM    = 1.0
MIXED_PREC       = True

# Checkpointing
# - latest.pth                  overwritten every epoch (resume target)
# - best_model.pth              overwritten on val_rmse improvement
# - checkpoint_epoch_NNNN.pth   every SAVE_EVERY epochs, NEVER deleted
SAVE_EVERY       = 1               # from 2 from 5 
PATIENCE         = 30              # from 10
BEST_METRIC      = "rmse"          # lower-is-better

# SGD-specific (recipe C only)
SGD_MOMENTUM     = 0.9
SGD_NESTEROV     = True

# Train/val split
VAL_FRAC         = 0.10     # from 0.05
SPLIT_SEED       = 42

# Dataset
TARGET_H         = 378      # multiple of 14
TARGET_W         = 1246     # multiple of 14
MIN_DEPTH        = 1e-3
MAX_DEPTH        = 80.0
NUM_WORKERS      = 16       # from 8
PERSISTENT_WORKERS = True    # keeps workers alive across epochs -> no shutdown noise
MAX_TRAIN        = 0
MAX_TEST         = 0
_pf              = 4        # Prefetch factor = force the CPU workers to eagerly load several batches in advance and keep them ready in RAM.

# Evaluation
MIN_MASK_SCORE   = 0.40
# SAM3 pre-generated mask root (KITTI format). Set to None to disable per-mask eval.
MASK_ROOT: Optional[str] = "/home/arka/Downloads/MMDE/outputs/inst_seg_output"
SAVE_NPY         = False

# Periodic checkpoint evaluation
# Number of test frames used per checkpoint for the epoch-RMSE curve plot.
# Set to 0 to evaluate the full test set (slower).
PERIODIC_TEST_SAMPLES = 0

# Loss recipes
LOSS_RECIPES: Dict[str, Dict] = {
    "A1": {"components": {"mae_canonical": 1.0, "silog": 1.0, "grad": 0.1},
           "optim": None, "lr": None},
    "A2": {"components": {"mae_canonical": 1.0, "silog": 1.0},
           "optim": None, "lr": None},
    "A3": {"components": {"mae_canonical": 1.0, "silog": 1.0,
                          "grad": 0.1, "berhu": 0.5},
           "optim": None, "lr": None},
    "A4": {"components": {"mae_canonical": 1.0, "silog": 1.0,
                          "grad": 0.1, "masked_depth": 0.5},
           "optim": None, "lr": None},
    "B1": {"components": {"mae_canonical": 0.5, "silog": 1.0,
                          "berhu": 1.0, "l1": 0.5, "grad": 0.1},
           "optim": None, "lr": None},
    "B2": {"components": {"mae_canonical": 1.0, "silog": 1.0,
                          "berhu": 1.0, "l1": 0.5, "grad": 0.1},
           "optim": None, "lr": None},
    "C":  {"components": {"mae_canonical": 1.0},
           "optim": "sgd", "lr": 1e-4},
}

# Loss component hyperparameters
SILOG_LAMBDA     = 0.85
BERHU_THRESHOLD  = 0.20
MAE_W_GRAD       = 0.5
MAE_N_SCALES     = 4
MASKED_LAMBDA_C  = 0.5

# MAECanonical depth floor for inverse-depth clamping.
# Default 0.1 m -> 1/pred <= 10, comparable to SILog (~0.04-0.5).
# Raise toward 0.5-1.0 m if loss imbalance persists.
MAE_DEPTH_FLOOR  = 0.5      # from 0.1

# Derived paths (one folder per recipe so runs never overwrite each other)
OUTPUT_BASE          = os.path.join(OUTPUT_BASE_ROOT, f"loss_{LOSS_NAME}")
CHECKPOINT_DIR       = os.path.join(OUTPUT_BASE, "checkpoints")
EVAL_OUTPUT_DIR      = os.path.join(OUTPUT_BASE, "eval_results")
HISTORY_JSON_PATH    = os.path.join(OUTPUT_BASE, "training_history.json")

PLOT_DIR             = os.path.join(EVAL_OUTPUT_DIR, "plots")
OUTPUT_CSV_PATH      = os.path.join(EVAL_OUTPUT_DIR, "DA3_finetuned_results.csv")
PERIODIC_CSV_PATH    = os.path.join(EVAL_OUTPUT_DIR, "periodic_test_metrics.csv")

if LOSS_NAME not in LOSS_RECIPES:
    raise ValueError(f"Unknown LOSS_NAME={LOSS_NAME!r}. Choose from: {list(LOSS_RECIPES)}")

# Execution flags
SKIP_TRAINING   = False   # True -> skip training, load existing best checkpoint
SKIP_INFERENCE  = False   # True -> skip inference, use existing CSV
SKIP_PLOTS      = False   # True -> summary only, no plot files
SKIP_PERIODIC   = False   # True -> skip per-epoch test_rmse evaluation
EVAL_START_IDX  = 0       # resume evaluation from this frame index

# ─────────────────────────────────────────────────────────────────────────────
EPS = 1e-6

## 2 · Logging Setup

In [4]:
# ─────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("da3_kitti")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ─────────────────────────────────────────────────────────────────────────────

## 3 · KITTI Eigen-Split Dataset Loader

Loads pre-projected depth PNGs directly — **no velodyne processing needed**.  
Supports random horizontal flip + colour jitter augmentation during training.

In [5]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from kitti_dataset import (
    KITTIEigenDepthDataset,
    collate_fn,
    _IMAGENET_MEAN,
    _IMAGENET_STD,
    _read_kitti_depth,        # used in run_evaluation
)

## 4 · DA3METRIC-LARGE Model Wrapper

- `load_da3_model` — loads base model (+ optional fine-tuned weights)  
- `configure_trainable_parameters` — option to opt for either decoder only or full architecture
- `run_da3_inference` — single-image inference with ortho→metric conversion  
- `forward_batch` — batch-level forward pass used during training

In [6]:
# ------------------------------------------------------------------------------------
#
# KEY FIX (applied throughout this cell):
# DepthAnything3 is a *wrapper* whose .forward() is decorated with
#   @torch.inference_mode()
# This makes ALL output tensors permanently non-differentiable.
# torch.enable_grad() / autocast CANNOT override it.
# The actual trainable network lives at model.model (DepthAnything3Net),
# which has no such decoration.
# Solution: call model.model(...) directly for training.
#
# ------------------------------------------------------------------------------------

_DA3_TRANSFORM = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])


def load_da3_model(
    device: torch.device,
    checkpoint_path: Optional[str] = None,
) -> nn.Module:
    """
    Load DA3METRIC-LARGE from HuggingFace Hub.
    If checkpoint_path is given, injects fine-tuned weights afterwards.

    NOTE: DepthAnything3.__init__ calls self.model.eval() internally.
    We do NOT call model.train() here; that is done in _unfreeze_decoder_only
    and in train_one_epoch so the mode is always explicit at point of use.
    """
    from depth_anything_3.api import DepthAnything3
    log.info("Loading %s ...", MODEL_ID)
    model = DepthAnything3.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
    model = model.to(device)

    if checkpoint_path and os.path.isfile(checkpoint_path):
        log.info("  Loading checkpoint weights: %s", checkpoint_path)
        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        sd   = ckpt.get("model_state_dict", ckpt)
        miss, unexp = model.load_state_dict(sd, strict=False)
        n_miss  = len([k for k in miss  if "num_batches" not in k])
        n_unexp = len([k for k in unexp if "num_batches" not in k])
        log.info("  Weights loaded -- missing: %d  unexpected: %d", n_miss, n_unexp)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info("  Parameters: %.1fM total, %.1fM trainable",
             total / 1e6, trainable / 1e6)
    return model


# def _unfreeze_decoder_only(model: nn.Module) -> None:
#     """
#     Freeze the backbone encoder; keep only the decoder and head trainable.
#     Also switches the inner DepthAnything3Net to train mode -- necessary
#     because DepthAnything3.__init__ calls self.model.eval() at construction.
#     """
#     for name, param in model.named_parameters():
#         if any(k in name for k in ["encoder", "pretrained", "backbone", "net.net"]):
#             param.requires_grad_(False)
#         else:
#             param.requires_grad_(True)

#     # Explicitly set the inner network to train mode.
#     # model.train() on the wrapper alone is NOT sufficient because
#     # DepthAnything3.__init__ hard-sets self.model.eval().
#     model.model.train()

#     trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
#     log.info("  After freeze-encoder: %.1fM trainable params", trainable / 1e6)

def configure_trainable_parameters(model: nn.Module,
                                   arch_mode: str = ARCH_MODE):
    """
    ARCH_MODE:
        decoder_only
        full
    """

    if arch_mode == "decoder_only":
        log.info("Architecture: Decoder-only fine-tuning")

        for name, p in model.named_parameters():
            if any(k in name for k in [
                "encoder",
                "pretrained",
                "backbone",
                "net.net"
            ]):
                p.requires_grad_(False)
            else:
                p.requires_grad_(True)

    elif arch_mode == "full":
        log.info("Architecture: Full-model fine-tuning")

        for p in model.parameters():
            p.requires_grad_(True)

    else:
        raise ValueError(f"Unknown ARCH_MODE: {arch_mode}")

    model.model.train()

    trainable = sum(p.numel() for p in model.parameters()
                    if p.requires_grad)

    total = sum(p.numel() for p in model.parameters())

    log.info(
        "Trainable parameters: %.2fM / %.2fM",
        trainable / 1e6,
        total / 1e6,
    )


def run_da3_inference(
    model: nn.Module,
    image_np: np.ndarray,
    device: torch.device,
    fx: float = FOCAL_LENGTH_X,
) -> np.ndarray:
    """
    Run DA3METRIC-LARGE on one uint8 RGB image (H x W x 3).
    Returns float32 metric depth map in metres, same spatial size as input.
    Uses the DepthAnything3 wrapper (inference_mode) -- fine for eval.
    """
    orig_h, orig_w = image_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)

    img_pil = Image.fromarray(image_np).resize((new_w, new_h), Image.Resampling.BICUBIC)
    t_img   = _DA3_TRANSFORM(img_pil)
    inp     = t_img.unsqueeze(0).unsqueeze(1).to(device)  # (1,1,3,H,W)

    model.model.eval()  # ensure eval mode on inner net
    with torch.no_grad():
        preds = model(inp, export_feat_layers=[])

    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
    elif isinstance(preds, (tuple, list)):
        raw = preds[0]
    else:
        raw = preds

    if isinstance(raw, torch.Tensor):
        if raw.dim() == 3:
            raw = raw.unsqueeze(1)
        depth = (fx * raw) / ORTHO_NORM
        depth = F.interpolate(depth, size=(orig_h, orig_w),
                              mode="bilinear", align_corners=False)
    else:
        raise RuntimeError(
            f"run_da3_inference: unexpected model output type {type(raw)}"
        )

    return depth.squeeze().cpu().float().numpy()


def forward_batch(
    model: nn.Module,
    rgb_batch: torch.Tensor,    # [B, 3, H, W]  normalised
    K_batch:   torch.Tensor,    # [B, 3, 3]
    has_K:     torch.Tensor,    # [B] bool
) -> torch.Tensor:
    """
    Differentiable forward pass returning metric depth [B, 1, H, W].

    CRITICAL: calls model.model (DepthAnything3Net) directly, bypassing
    the @torch.inference_mode() decorator on the DepthAnything3 wrapper.
    That decorator makes all output tensors permanently non-differentiable
    and CANNOT be overridden by torch.enable_grad() or autocast.
    """
    B, C, H, W = rgb_batch.shape
    inp = rgb_batch.unsqueeze(1)   # (B, 1, 3, H, W)

    # Call the inner DepthAnything3Net directly (no inference_mode)
    preds = model.model(inp, export_feat_layers=[])

    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
        # Replace the intrinsics branch in forward_batch with:
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X, device=rgb_batch.device, dtype=rgb_batch.dtype)
        
    elif isinstance(preds, (tuple, list)):
        raw     = preds[0]
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)
    else:
        raw     = preds
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)

    if raw.dim() == 3:
        raw = raw.unsqueeze(1)

    # Ortho -> metric
    depth = (fx_pred * raw) / ORTHO_NORM

    if depth.shape[-2:] != (H, W):
        depth = F.interpolate(depth, size=(H, W),
                              mode="bilinear", align_corners=False)
    return depth   # [B, 1, H, W] -- carries grad_fn through model.model params


# ------------------------------------------------------------------------------------


## 5 · Loss Functions

Weighted combination of:
- **SILog** (Scale-Invariant Log) — main depth supervision  
- **BerHu** (Reverse Huber) — robust to outliers  
- **L1** — absolute depth error  
- **GradientLoss** — edge sharpness

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Loss functions -- 7 recipes via LOSS_NAME.
#
# KEY FIX: MAECanonical now clamps pred/gt to [MAE_DEPTH_FLOOR, MAX_DEPTH]
# before inversion. With default MAE_DEPTH_FLOOR=0.5m, keeping the loss comparable in magnitude
# to SILog/BerHu/GradientLoss so recipe weights actually take effect.
# ─────────────────────────────────────────────────────────────────────────────



def _safe_log(x: torch.Tensor) -> torch.Tensor:
    return torch.log(x.clamp(min=EPS))


class SILogLoss(nn.Module):
    """Scale-Invariant Log loss (Eigen et al., NeurIPS 2014)."""
    def __init__(self, lam: float = SILOG_LAMBDA):
        super().__init__()
        self.lam = lam
    def forward(self, pred, gt, mask):
        pred = pred.squeeze(1); gt = gt.squeeze(1); mask = mask.squeeze(1).bool()
        if not mask.any(): return pred.sum() * 0.0
        d = _safe_log(pred[mask]) - _safe_log(gt[mask])
        return torch.sqrt((d**2).mean() - self.lam * d.mean()**2 + EPS)


class BerHuLoss(nn.Module):
    """Reverse-Huber (BerHu) loss -- Laina et al., 3DV 2016."""
    def __init__(self, threshold: float = BERHU_THRESHOLD):
        super().__init__()
        self.c = threshold
    def forward(self, pred, gt, mask):
        pred = pred.squeeze(1); gt = gt.squeeze(1); mask = mask.squeeze(1).bool()
        if not mask.any(): return pred.sum() * 0.0
        diff = (pred - gt).abs()[mask]
        c    = (self.c * diff.max().detach()).clamp(min=EPS)
        l1   = diff[diff <= c]; l2 = diff[diff > c]
        loss = l1.sum()
        if l2.numel() > 0: loss = loss + ((l2**2 + c**2) / (2*c)).sum()
        return loss / diff.numel()


class L1DepthLoss(nn.Module):
    """Plain masked L1 in metric depth space."""
    def forward(self, pred, gt, mask):
        return ((pred - gt).abs() * mask).sum() / (mask.sum() + EPS)


class GradientLoss(nn.Module):
    """Image-gradient-matching loss in metric depth space."""
    def forward(self, pred, gt, mask):
        def _grad(x): return x[...,:-1]-x[...,1:], x[...,:-1,:]-x[...,1:,:]
        m = mask.float()
        p_dx, p_dy = _grad(pred*m); g_dx, g_dy = _grad(gt*m)
        mx_x = m[...,:-1]*m[...,1:]; mx_y = m[...,:-1,:]*m[...,1:,:]
        # return (((p_dx-g_dx).abs()*mx_x).mean() + ((p_dy-g_dy).abs()*mx_y).mean())
        # FIX: Average only over the valid pixel pairs
        loss_x = ((p_dx - g_dx).abs() * mx_x).sum() / (mx_x.sum() + 1e-6)
        loss_y = ((p_dy - g_dy).abs() * mx_y).sum() / (mx_y.sum() + 1e-6)
        
        return loss_x + loss_y


class MAECanonicalLoss(nn.Module):
    """
    DepthPro L_MAE in canonical inverse-depth space + multi-scale gradient pyramid.

    FIXED: pred/gt clamped to [MAE_DEPTH_FLOOR, MAX_DEPTH] before inversion.
    With MAE_DEPTH_FLOOR=0.5 m.
    This keeps the loss magnitude comparable to SILog/BerHu so recipe weights
    actually balance the components instead of mae_canonical dominating by
    orders of magnitude due to near-zero depth predictions.

    Tune MAE_DEPTH_FLOOR in Section 1 if imbalance persists.
    """
    def __init__(self, w_grad: float = MAE_W_GRAD, n_scales: int = MAE_N_SCALES,
                 depth_floor: float = MAE_DEPTH_FLOOR):
        super().__init__()
        self.w_grad, self.n_scales, self.depth_floor = w_grad, n_scales, depth_floor
        self.grad = GradientLoss()

    def forward(self, pred, gt, mask):
        p  = pred.clamp(self.depth_floor, MAX_DEPTH)   # tighter floor
        g  = gt.clamp(self.depth_floor,   MAX_DEPTH)
        ip = 1.0 / p    # bounded: [1/MAX_DEPTH, 1/depth_floor]
        ig = 1.0 / g
        m  = mask.float()
        l_mae = ((ip-ig).abs() * m).sum() / (m.sum() + EPS)
        l_grad = torch.zeros((), device=pred.device, dtype=pred.dtype)
        pc, gc, mc = ip, ig, m
        for s in range(self.n_scales):
            l_grad = l_grad + self.grad(pc, gc, mc.gt(0.5).float())
            if s < self.n_scales - 1:
                # pc = F.avg_pool2d(pc, 2, stride=2)
                # gc = F.avg_pool2d(gc, 2, stride=2)
                # mc = F.avg_pool2d(mc, 2, stride=2)
                mc_down = F.avg_pool2d(mc, 2, stride=2)
                # Multiply by mask BEFORE pooling so invalid pixels become true 0s
                # Divide by mc_down to get the proper average of ONLY valid pixels
                pc = F.avg_pool2d(pc * mc, 2, stride=2) / (mc_down + EPS)
                gc = F.avg_pool2d(gc * mc, 2, stride=2) / (mc_down + EPS)
                mc = mc_down
        l_grad = l_grad / self.n_scales
        return l_mae + self.w_grad * l_grad


class MaskedDepthLoss(nn.Module):
    def __init__(
        self,
        min_depth: float = MAE_DEPTH_FLOOR,
        max_depth: float = MAX_DEPTH,
        eps: float = 1e-6,
    ):
        super().__init__()
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.eps = eps

    def forward(self, pred, gt, mask):
        m = mask.bool()
        if not m.any():
            return pred.sum() * 0.0
        # No clamp on p_valid to preserve gradients!
        # (It is already safely clamped to EPS by CompositeLoss)
        p_valid = pred[m]  
        # Clamp g_valid to prevent infinite weights
        g_valid = gt[m].clamp(min=self.min_depth, max=self.max_depth)
        # Inverse-depth weights
        weights = 1.0 / g_valid
        # Sum-normalize weights so they sum to 1.0
        weights = weights / (weights.sum().detach() + self.eps)
        # Apply sum aggregation
        loss = (weights * (p_valid - g_valid).abs()).sum()
        return loss


def _build_component(name: str) -> nn.Module:
    if name == "silog":         return SILogLoss()
    if name == "berhu":         return BerHuLoss()
    if name == "l1":            return L1DepthLoss()
    if name == "grad":          return GradientLoss()
    if name == "mae_canonical": return MAECanonicalLoss()
    if name == "masked_depth":  return MaskedDepthLoss()
    raise ValueError(f"Unknown loss component: {name!r}")


class CompositeLoss(nn.Module):
    """Weighted sum of named components. Returns (total, losses_dict)."""
    def __init__(self, weights: Dict[str, float], recipe_name: str = ""):
        super().__init__()
        self.recipe_name = recipe_name
        self.names   = list(weights.keys())
        self.weights = [float(weights[n]) for n in self.names]
        self.components = nn.ModuleList([_build_component(n) for n in self.names])
    def forward(self, pred, gt, mask) -> Tuple[torch.Tensor, Dict]:
        pred_c = pred.clamp(EPS, MAX_DEPTH)
        losses: Dict[str, torch.Tensor] = {}
        total  = torch.zeros((), device=pred.device, dtype=pred.dtype)
        for w, name, mod in zip(self.weights, self.names, self.components):
            l = mod(pred_c, gt, mask)
            losses[name] = l
            total = total + w * l
        losses["total"] = total
        return total, losses


def build_loss(name: str = LOSS_NAME) -> nn.Module:
    weights = LOSS_RECIPES[name]["components"]
    pretty  = " + ".join(f"{w:g}*{k}" for k, w in weights.items())
    log.info("Loss recipe %s : %s", name, pretty)
    log.info("  MAECanonical depth_floor=%.3f m  (1/pred <= %.1f)",
             MAE_DEPTH_FLOOR, 1.0/MAE_DEPTH_FLOOR)
    return CompositeLoss(weights, recipe_name=name)
# ─────────────────────────────────────────────────────────────────────────────


## 6 · Checkpoint Utilities

Supports **full training resume**:  
- Saves model weights + optimizer + scheduler + scaler state  
- Auto-detects latest checkpoint in `CHECKPOINT_DIR`  
- Keeps only the last N checkpoints (configurable)  
- Separately saves `best_model.pth` based on validation AbsRel

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint strategy -- three distinct files, no pruning of periodic ones:
#   latest.pth                  overwritten every epoch (primary resume target)
#   best_model.pth              overwritten on val_rmse improvement
#   checkpoint_epoch_NNNN.pth   saved every SAVE_EVERY epochs, NEVER deleted
#
# Training history (epoch metrics) is saved to HISTORY_JSON_PATH and reloaded
# on resume so epoch-curve plots survive across restarts.
# ─────────────────────────────────────────────────────────────────────────────

def _make_state(model, optimizer, scaler, scheduler,
                epoch, global_step, best_metric, val_metrics, patience_ctr):
    return {
        "epoch": epoch, "global_step": global_step,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "best_metric":    best_metric,
        "best_metric_name": BEST_METRIC,
        "val_metrics":    val_metrics,
        "patience_counter": patience_ctr,
        "loss_name":      LOSS_NAME,
    }


def save_latest(model, optimizer, scaler, scheduler,
                epoch, global_step, best_metric, val_metrics, patience_ctr, out_dir):
    """Overwrite latest.pth -- used to resume training from exactly where it stopped."""
    os.makedirs(out_dir, exist_ok=True)
    torch.save(_make_state(model, optimizer, scaler, scheduler, epoch,
                           global_step, best_metric, val_metrics, patience_ctr),
               os.path.join(out_dir, "latest.pth"))


def save_periodic(model, optimizer, scaler, scheduler,
                  epoch, global_step, best_metric, val_metrics, patience_ctr, out_dir):
    """Save checkpoint_epoch_NNNN.pth -- kept forever, used for epoch-RMSE curves."""
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"checkpoint_epoch_{epoch:04d}.pth")
    torch.save(_make_state(model, optimizer, scaler, scheduler, epoch,
                           global_step, best_metric, val_metrics, patience_ctr), path)
    log.info("  Periodic ckpt saved -> %s", path)
    return path


def save_best(model, optimizer, scaler, scheduler,
              epoch, global_step, best_metric, val_metrics, patience_ctr, out_dir):
    """Overwrite best_model.pth when val_rmse improves."""
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "best_model.pth")
    torch.save(_make_state(model, optimizer, scaler, scheduler, epoch,
                           global_step, best_metric, val_metrics, patience_ctr), path)
    log.info("  New best model (val_%s=%.4f) -> %s", BEST_METRIC, best_metric, path)


def find_resume_checkpoint(out_dir: str) -> Optional[str]:
    """Prefer latest.pth; fall back to highest checkpoint_epoch_*.pth."""
    lp = os.path.join(out_dir, "latest.pth")
    if os.path.exists(lp): return lp
    periodic = sorted(glob.glob(os.path.join(out_dir, "checkpoint_epoch_*.pth")))
    return periodic[-1] if periodic else None


def list_periodic_checkpoints(out_dir: str) -> List[str]:
    return sorted(glob.glob(os.path.join(out_dir, "checkpoint_epoch_*.pth")))


def load_checkpoint(ckpt_path, model, optimizer, scaler, scheduler) -> Dict:
    log.info("Resuming from: %s", ckpt_path)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"], strict=False)
    try:    
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        # FORCE the new learning rate into the loaded optimizer state
        for param_group in optimizer.param_groups:
            param_group['lr'] = LEARNING_RATE
    except Exception as e: 
        log.warning("Optimizer state not loaded: %s", e)
    # Leave the scheduler load COMMENTED OUT. 
    # Generating a fresh scheduler is fine as long as the optimizer keeps its historical variance.
    try:    scaler.load_state_dict(ckpt.get("scaler_state_dict", scaler.state_dict()))
    except Exception: pass

    best = ckpt.get("best_metric", ckpt.get("best_abs_rel", float("inf")))
    return {"epoch": ckpt.get("epoch", 0), "global_step": ckpt.get("global_step", 0),
            "best_metric": best, "patience_counter": ckpt.get("patience_counter", 0)}


# Training history helpers
def load_history() -> List[Dict]:
    if os.path.exists(HISTORY_JSON_PATH):
        with open(HISTORY_JSON_PATH) as f: return json.load(f)
    return []

def append_history(history: List[Dict], record: Dict) -> None:
    history = [h for h in history if h.get("epoch") != record["epoch"]]
    history.append(record)
    history.sort(key=lambda h: h["epoch"])
    os.makedirs(os.path.dirname(HISTORY_JSON_PATH) or ".", exist_ok=True)
    with open(HISTORY_JSON_PATH, "w") as f: json.dump(history, f, indent=2)
# ─────────────────────────────────────────────────────────────────────────────


## 7 · Optimizer & Scheduler

AdamW with cosine annealing + warmup.

In [9]:
# ─────────────────────────────────────────────────────────────────────────────

def build_optimizer(model: nn.Module,
                    optim_mode: str = OPTIM_MODE,
                    lr:         float = LEARNING_RATE) -> torch.optim.Optimizer:
    """
    Default: AdamW (LR=2e-5, WD=1e-4).
    Recipe C overrides to SGD + Nesterov momentum (LR=1e-4, see LOSS_RECIPES).
    """
    trainable = list(filter(lambda p: p.requires_grad, model.parameters()))
    if optim_mode == "sgd":
        log.info("Optimizer: SGD  lr=%.1e  momentum=%g  nesterov=%s  wd=%.1e",
                 lr, SGD_MOMENTUM, SGD_NESTEROV, WEIGHT_DECAY)
        return torch.optim.SGD(
            trainable, lr=lr,
            momentum=SGD_MOMENTUM, weight_decay=WEIGHT_DECAY,
            nesterov=SGD_NESTEROV,
        )
    log.info("Optimizer: AdamW  lr=%.1e  wd=%.1e", lr, WEIGHT_DECAY)
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)


def build_scheduler(
    optimizer:        torch.optim.Optimizer,
    steps_per_epoch:  int,
    base_lr:          float = LEARNING_RATE,
):
    """
    Linear warmup → cosine annealing (per optimizer step).
    `base_lr` is used to set eta_min so it tracks recipe-specific LR overrides.
    """
    from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR

    warm_steps = WARMUP_EPOCHS * steps_per_epoch
    main_steps = (EPOCHS - WARMUP_EPOCHS) * steps_per_epoch
    log.info("Scheduler: cosine_warmup  warm_steps=%d  main_steps=%d  eta_min=%.1e",
             warm_steps, main_steps, base_lr * 0.01)

    warmup = LambdaLR(
        optimizer,
        lr_lambda=lambda s: min(1.0, (s + 1) / max(warm_steps, 1)),
    )
    cosine = CosineAnnealingLR(
        optimizer, T_max=max(main_steps, 1),
        eta_min=base_lr * 0.01,
    )
    return SequentialLR(optimizer, [warmup, cosine], milestones=[warm_steps])


# ─────────────────────────────────────────────────────────────────────────────


## 8 · Training & Validation Loop

- Gradient accumulation (configurable via `ACCUM_STEPS`)  
- Mixed-precision (AMP) training  
- Gradient clipping  
- Per-step learning rate scheduling

In [10]:
# ------------------------------------------------------------------------------------

def train_one_epoch(model, loader, criterion, optimizer, scaler,
                    scheduler, device, epoch, global_step,
                    log_every: int = 20) -> Tuple[int, float]:
    model.model.train()
    running: Dict[str, float] = {}
    epoch_loss = 0.0
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)

    for bi, batch in enumerate(loader):
        rgb   = batch["rgb"].to(device, non_blocking=True)
        gt    = batch["depth_gt"].to(device, non_blocking=True)
        mask  = batch["valid_mask"].to(device, non_blocking=True)
        K     = batch["K"].to(device, non_blocking=True)
        has_K = batch["has_K"].to(device, non_blocking=True)

        with autocast(device_type="cuda",
                      enabled=MIXED_PREC and torch.cuda.is_available()):
            pred          = forward_batch(model, rgb, K, has_K)
            total, losses = criterion(pred, gt, mask)
            total_accum   = total / ACCUM_STEPS

        scaler.scale(total_accum).backward()

        if (bi + 1) % ACCUM_STEPS == 0:
            if MAX_GRAD_NORM > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    (p for p in model.parameters() if p.requires_grad),
                    MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler: scheduler.step()
            global_step += 1

        epoch_loss += float(total.detach())
        for k, v in losses.items():
            running[k] = running.get(k, 0.0) + float(v.detach())

        if (bi + 1) % log_every == 0:
            avg   = {k: v/log_every for k, v in running.items()}
            lr    = optimizer.param_groups[0]["lr"]
            extra = "  ".join(f"{k}={v:.4f}" for k,v in avg.items() if k != "total")
            log.info("Ep[%d] Batch[%d/%d]  Loss=%.4f  %s  LR=%.2e  T=%.1fs",
                     epoch, bi+1, len(loader),
                     avg.get("total", 0.0), extra, lr, time.time()-t0)
            running = {}; t0 = time.time()

    return global_step, epoch_loss / max(len(loader), 1)


@torch.no_grad()
def validate(model, loader, criterion, device, epoch) -> Dict[str, float]:
    model.model.eval()
    total_loss = 0.0
    sum_sq_err = 0.0; n_valid_total = 0
    all_abs_rel: List[float] = []
    all_sq_rel:  List[float] = []
    all_rmse_log:List[float] = []
    all_d1: List[float] = []; all_d2: List[float] = []; all_d3: List[float] = []

    for batch in tqdm(loader, desc=f"Val Ep{epoch}", leave=False):
        rgb   = batch["rgb"].to(device, non_blocking=True)
        gt    = batch["depth_gt"].to(device, non_blocking=True)
        mask  = batch["valid_mask"].to(device, non_blocking=True)
        K     = batch["K"].to(device, non_blocking=True)
        has_K = batch["has_K"].to(device, non_blocking=True)
        with autocast(device_type="cuda",
                      enabled=MIXED_PREC and torch.cuda.is_available()):
            pred    = forward_batch(model, rgb, K, has_K)
            loss, _ = criterion(pred, gt, mask)
        total_loss += float(loss)
        m = compute_torch_metrics(pred, gt, mask)
        sum_sq_err    += m["sum_sq_err"]
        n_valid_total += m["n_valid"]
        if m["n_valid"] > 0:
            all_abs_rel.append(m["abs_rel"]); all_sq_rel.append(m["sq_rel"])
            all_rmse_log.append(m["rmse_log"])
            all_d1.append(m["d1"]); all_d2.append(m["d2"]); all_d3.append(m["d3"])

    pw = float(np.sqrt(sum_sq_err / max(n_valid_total, 1)))
    def _m(lst): return float(np.mean(lst)) if lst else float("nan")
    agg = {"rmse": pw, "abs_rel": _m(all_abs_rel), "sq_rel": _m(all_sq_rel),
           "rmse_log": _m(all_rmse_log), "d1": _m(all_d1), "d2": _m(all_d2),
           "d3": _m(all_d3), "val_loss": total_loss / max(len(loader), 1)}
    log.info("[VAL] Ep%d  Loss=%.4f  RMSE(pw)=%.4f  AbsRel=%.4f  d1=%.4f",
             epoch, agg["val_loss"], agg["rmse"], agg["abs_rel"], agg["d1"])
    return agg


def compute_torch_metrics(pred, gt, mask) -> Dict:
    """Per-batch metrics with sum_sq_err/n_valid for pixel-weighted RMSE."""
    pred_c = pred.clamp(EPS, MAX_DEPTH)
    mb     = mask.bool()
    if not mb.any():
        return {k: float("nan") for k in
                ["abs_rel","sq_rel","rmse","rmse_log","d1","d2","d3",
                 "sum_sq_err","n_valid"]} | {"sum_sq_err": 0.0, "n_valid": 0}
    p = pred_c[mb].float(); g = gt[mb].float().clamp(EPS)
    sq_err = (g-p)**2; thresh = torch.maximum(g/p, p/g)
    return {
        "abs_rel":   float(((g-p).abs()/g).mean()),
        "sq_rel":    float((sq_err/g).mean()),
        "rmse":      float(torch.sqrt(sq_err.mean())),
        "rmse_log":  float(torch.sqrt((_safe_log(p)-_safe_log(g)).pow(2).mean())),
        "d1":        float((thresh < 1.25).float().mean()),
        "d2":        float((thresh < 1.25**2).float().mean()),
        "d3":        float((thresh < 1.25**3).float().mean()),
        "sum_sq_err":float(sq_err.sum()),
        "n_valid":   int(mb.sum()),
    }
# ------------------------------------------------------------------------------------


## 9 · Main Training Function  (`run_training`)

Orchestrates the full training loop with resume support.  
Set `skip_training=True` in the final execution cell to skip this step.

In [11]:
# ─────────────────────────────────────────────────────────────────────────────

def _split_train_val(train_root, val_frac=VAL_FRAC, seed=SPLIT_SEED,
                     aug_mode=AUG_MODE, max_train=MAX_TRAIN):
    log.info("Building 90/10 train-val split (val_frac=%.2f, seed=%d)", val_frac, seed)
    kw = dict(split="train", max_samples=max_train, seed=seed)
    train_full = KITTIEigenDepthDataset(train_root, augment=True,  **kw)
    val_full   = KITTIEigenDepthDataset(train_root, augment=False, **kw)
    n = len(train_full); n_val = max(1, int(round(n * val_frac)))
    rng  = np.random.RandomState(seed)
    perm = rng.permutation(n).tolist()
    val_idx, train_idx = perm[:n_val], perm[n_val:]
    train_sub = torch.utils.data.Subset(train_full, train_idx)
    val_sub   = torch.utils.data.Subset(val_full,   val_idx)
    log.info("  train=%d  val=%d  (total %d)", len(train_sub), len(val_sub), n)
    return train_sub, val_sub


def run_training(device: torch.device) -> str:
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_BASE,    exist_ok=True)

    recipe     = LOSS_RECIPES[LOSS_NAME]
    optim_mode = recipe.get("optim") or OPTIM_MODE
    lr         = recipe.get("lr")    or LEARNING_RATE
    log.info("Recipe: %s  optim=%s  lr=%.1e  mae_floor=%.3f",
             LOSS_NAME, optim_mode, lr, MAE_DEPTH_FLOOR)

    train_ds, val_ds = _split_train_val(TRAIN_ROOT)

    # persistent_workers=True keeps worker processes alive between epochs,
    # eliminating the "can only test a child process" shutdown AssertionError.
    _pw = PERSISTENT_WORKERS and NUM_WORKERS > 0
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
        collate_fn=collate_fn, persistent_workers=_pw, prefetch_factor=_pf)
    val_loader = DataLoader(
        val_ds, batch_size=1, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        collate_fn=collate_fn, persistent_workers=_pw, prefetch_factor=_pf)
    log.info("Train: %d batches | Val: %d batches", len(train_loader), len(val_loader))

    history  = load_history()
    model    = load_da3_model(device, checkpoint_path=None)
    # _unfreeze_decoder_only(model)
    configure_trainable_parameters(
    model,
    arch_mode=ARCH_MODE
    )
    
    criterion = build_loss(LOSS_NAME).to(device)
    optimizer = build_optimizer(model, optim_mode=optim_mode, lr=lr)
    scheduler = build_scheduler(optimizer, len(train_loader)// ACCUM_STEPS, base_lr=lr) #divide the steps by ACCUM_STEPS so the math aligns with how often you call scheduler.step()
    scaler    = GradScaler("cuda" if torch.cuda.is_available() else "cpu",
                           enabled=MIXED_PREC)

    # Restore state
    start_epoch = 1; global_step = 0
    best_val    = float("inf"); patience_ctr = 0

    resume = find_resume_checkpoint(CHECKPOINT_DIR)
    if resume:
        meta         = load_checkpoint(resume, model, optimizer, scaler, scheduler)
        start_epoch  = meta["epoch"] + 1
        global_step  = meta["global_step"]
        best_val     = meta["best_metric"]
        patience_ctr = meta.get("patience_counter", 0)
        log.info("Resumed from epoch %d  best val_%s=%.4f  patience=%d/%d",
                 start_epoch, BEST_METRIC, best_val, patience_ctr, PATIENCE)
    else:
        log.info("Fresh training (no checkpoint found)")

    _save_training_config(start_epoch, resume, optim_mode, lr)

    log.info("=" * 70)
    log.info("Loss=%s  Arch=%s  Aug=%s  Optim=%s", LOSS_NAME, ARCH_MODE, AUG_MODE, optim_mode)
    log.info("Epochs: %d->%d  BS=%dx%d=%d  LR=%.1e  MAE_floor=%.3f m",
             start_epoch, EPOCHS, BATCH_SIZE, ACCUM_STEPS,
             BATCH_SIZE*ACCUM_STEPS, lr, MAE_DEPTH_FLOOR)
    log.info("AMP=%s  Clip=%.1f  Patience=%d  SaveEvery=%d  Device=%s",
             MIXED_PREC, MAX_GRAD_NORM, PATIENCE, SAVE_EVERY, device)
    log.info("=" * 70)

    for epoch in range(start_epoch, EPOCHS + 1):
        log.info("─" * 60)
        log.info("EPOCH %d / %d", epoch, EPOCHS)

        global_step, train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer,
            scaler, scheduler, device, epoch, global_step)
        val_metrics = validate(model, val_loader, criterion, device, epoch)
        cur         = val_metrics[BEST_METRIC]

        is_best = cur < best_val
        if is_best:
            best_val = cur; patience_ctr = 0
            save_best(model, optimizer, scaler, scheduler, epoch, global_step,
                      best_val, val_metrics, patience_ctr, CHECKPOINT_DIR)
        else:
            patience_ctr += 1
            log.info("  No improvement %d/%d", patience_ctr, PATIENCE)

        # Always update latest.pth (resumability)
        save_latest(model, optimizer, scaler, scheduler, epoch, global_step,
                    best_val, val_metrics, patience_ctr, CHECKPOINT_DIR)

        # Periodic checkpoint (every SAVE_EVERY epochs, never pruned)
        if epoch % SAVE_EVERY == 0:
            save_periodic(model, optimizer, scaler, scheduler, epoch, global_step,
                          best_val, val_metrics, patience_ctr, CHECKPOINT_DIR)

        # Record training history
        rec = {"epoch": epoch, "train_loss": train_loss,
               "val_loss": val_metrics["val_loss"], "val_rmse": val_metrics["rmse"],
               "val_abs_rel": val_metrics["abs_rel"], "val_d1": val_metrics["d1"],
               "is_best": is_best}
        append_history(history, rec)
        history = load_history()

        if patience_ctr >= PATIENCE:
            log.info("Early stopping: no val_%s improvement for %d epochs (best=%.4f).",
                     BEST_METRIC, PATIENCE, best_val)
            break

    log.info("Training complete. Best val_%s=%.4f", BEST_METRIC, best_val)
    return os.path.join(CHECKPOINT_DIR, "best_model.pth")


def _save_training_config(start_epoch, resumed_from, optim_mode, lr):
    cfg = {"loss_name": LOSS_NAME, "arch_mode": ARCH_MODE, "aug_mode": AUG_MODE,
           "optim_mode": optim_mode, "lr": lr, "weight_decay": WEIGHT_DECAY,
           "epochs": EPOCHS, "batch_size": BATCH_SIZE, "accum_steps": ACCUM_STEPS,
           "effective_batch": BATCH_SIZE*ACCUM_STEPS,
           "warmup_epochs": WARMUP_EPOCHS, "max_grad_norm": MAX_GRAD_NORM,
           "mixed_prec": MIXED_PREC, "mae_depth_floor": MAE_DEPTH_FLOOR,
           "save_every": SAVE_EVERY, "patience": PATIENCE,
           "val_frac": VAL_FRAC, "split_seed": SPLIT_SEED,
           "target_hw": [TARGET_H, TARGET_W],
           "start_epoch": start_epoch, "resumed_from": resumed_from}
    with open(os.path.join(OUTPUT_BASE, "training_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)
# ─────────────────────────────────────────────────────────────────────────────


## 10 · Evaluation Analysis Helpers

Standard KITTI depth metrics and:
- **Unscaled** — raw model output  

Also includes optional per-mask evaluation using pre-generated SAM3 masks.

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# Evaluation helpers -- UNSCALED METRICS ONLY.
# Pixel-weighted RMSE across a group:
#   pw_rmse(df) = sqrt( sum(sum_sq_err) / sum(n_pixels) )
# ─────────────────────────────────────────────────────────────────────────────

def eigen_crop_mask(shape: Tuple[int, int]) -> np.ndarray:
    h, w = shape[:2]; m = np.zeros((h, w), dtype=bool)
    m[44:min(44+352, h), :] = True; return m


def compute_depth_errors(gt: np.ndarray, pred: np.ndarray,
                         mask: np.ndarray) -> Dict:
    """
    Unscaled KITTI depth metrics for pixels in `mask`.
    Returns per-row scalars PLUS sum_sq_err and n_pixels for pixel-weighted
    RMSE aggregation across rows (pw_rmse function).
    """
    gt_v   = gt[mask]
    pred_v = np.clip(pred[mask], 1e-8, None)
    n      = len(gt_v)
    if n == 0:
        nan = float("nan")
        return {"rmse": nan, "abs_rel": nan, "sq_rel": nan, "rmse_log": nan,
                "delta1": nan, "delta2": nan, "delta3": nan,
                "avg_pred_depth": nan, "sum_sq_err": 0.0, "n_pixels": 0}
    sq_err = (gt_v - pred_v)**2
    thresh = np.maximum(gt_v/pred_v, pred_v/gt_v)
    return dict(
        rmse          = float(np.sqrt(sq_err.mean())),
        abs_rel       = float(np.mean(np.abs(gt_v-pred_v)/gt_v)),
        sq_rel        = float(np.mean(sq_err/gt_v)),
        rmse_log      = float(np.sqrt(np.mean((np.log(gt_v)-np.log(pred_v))**2))),
        delta1        = float(np.mean(thresh < 1.25)),
        delta2        = float(np.mean(thresh < 1.25**2)),
        delta3        = float(np.mean(thresh < 1.25**3)),
        avg_pred_depth= float(pred_v.mean()),
        sum_sq_err    = float(sq_err.sum()),    # for pixel-weighted RMSE
        n_pixels      = n,                       # denominator for pixel-weighted RMSE
    )


def pw_rmse(df: pd.DataFrame) -> float:
    """Pixel-weighted RMSE: sqrt(sum(SSE) / sum(N_pixels)) over all rows in df."""
    sse = df["sum_sq_err"].sum(); npx = df["n_pixels"].sum()
    return float(np.sqrt(sse/npx)) if npx > 0 else float("nan")


# Mask helpers
def get_mask_dir(rgb_path: str, mask_root: str, kitti_root: str) -> str:
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), kitti_root)
    return os.path.join(mask_root, rel_data_dir, frame_stem)


def load_masks_for_frame(mask_dir: str) -> List[Dict]:
    p = Path(mask_dir)
    if not p.exists(): return []
    stem = p.name; manifest = p / f"{stem}_manifest.json"
    instances = []
    if manifest.exists():
        for rec in json.loads(manifest.read_text()):
            mf = p / rec["mask_file"]
            if not mf.exists(): continue
            instances.append({
                "uid": rec["uid"], "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"], "sub_cat": rec["sub_cat"],
                "score": float(rec["score"]), "box": rec.get("box"),
                "mask": np.array(Image.open(mf).convert("L")) > 127, "path": str(mf),
            })
        return instances
    for mf in sorted(p.glob("*_mask.png")):
        s = mf.stem[:-5]
        if not s.startswith(stem+"_"): continue
        parts = s[len(stem)+1:].rsplit("_", 3)
        if len(parts) != 4: continue
        cat_str, idx_str, uid_str, score_str = parts
        cp = cat_str.split("_", 1)
        if len(cp) != 2: continue
        super_cat, sub_cat = cp
        try:
            local_idx = int(idx_str); score = float(score_str)
        except ValueError: continue
        instances.append({
            "uid": uid_str, "local_idx": local_idx, "super_cat": super_cat,
            "sub_cat": sub_cat, "score": score, "box": None,
            "mask": np.array(Image.open(mf).convert("L")) > 127, "path": str(mf),
        })
    instances.sort(key=lambda x: x["local_idx"]); return instances
# ─────────────────────────────────────────────────────────────────────────────


## 11 · CSV Column Schema

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Unscaled metrics only. row_type = "fullframe" | "mask".
# sum_sq_err and n_pixels enable pixel-weighted RMSE aggregation.
# ─────────────────────────────────────────────────────────────────────────────
CSV_COLUMNS = [
    "img_id", "file_name", "row_type",
    "mask_id", "mask_uid", "mask_score",
    "super_cat", "sub_cat", "horizontal_angle",
    "rmse", "abs_rel", "sq_rel", "rmse_log",
    "delta1", "delta2", "delta3", "avg_pred_depth",
    "sum_sq_err", "n_pixels",
    "lidar_avg_depth", "lidar_min_depth",
    "rgb_image_path", "gt_depth_map_path",
]
# ─────────────────────────────────────────────────────────────────────────────


## 12 · Evaluation / Inference Loop

Iterates over the test split:
1. Loads RGB + LiDAR GT depth  
2. Runs fine-tuned DA3METRIC-LARGE inference  
3. Computes all four scale variants  
4. Appends row to CSV (incremental — supports resume via `eval_start_idx`)

In [14]:
# ─────────────────────────────────────────────────────────────────────────────

def index_kitti_test(test_root: str, use_train_too: bool = False,
                     train_root: Optional[str] = None) -> pd.DataFrame:
    rows = []
    def _collect(sr, tag):
        gp = glob.glob(os.path.join(sr,"*","*_sync","proj_depth","groundtruth","image_02","*.png"))
        if not gp:
            gp = glob.glob(os.path.join(sr,"*","*","proj_depth","groundtruth","image_02","*.png"))
        for gt_path in tqdm(gp, desc=f"Indexing {tag}"):
            parts = gt_path.split(os.sep)
            try:
                gi=parts.index("proj_depth"); drive=parts[gi-1]; date=parts[gi-2]; fname=parts[-1]
            except (ValueError,IndexError): continue
            rgb = os.path.join(sr, date, drive, "image_02", "data", fname)
            if not os.path.exists(rgb): continue
            rows.append(dict(folder=tag,sequence=date,drive=drive,filename=fname,
                             gt_path=gt_path,rgb_path=rgb))
    _collect(test_root,"test")
    if use_train_too and train_root: _collect(train_root,"train")
    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Test index: %d frames, %d drives", len(df), df["drive"].nunique())
    return df


def run_evaluation(model, test_index, device, start_index=0):
    """
    Run inference + metrics on every test frame.

    For each frame TWO types of rows are emitted:
      row_type="fullframe"  -- Eigen-crop eval, ALWAYS written (even if masks exist)
      row_type="mask"       -- per-SAM3-instance, written only when MASK_ROOT is set

    This gives a clear separation between full-frame and per-category metrics.
    """
    os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True); os.makedirs(PLOT_DIR, exist_ok=True)
    if not os.path.exists(OUTPUT_CSV_PATH) or start_index == 0:
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(OUTPUT_CSV_PATH, index=False)
    model.eval()

    for idx, row in tqdm(test_index.iterrows(), total=len(test_index),
                         desc="Evaluating"):
        if idx < start_index: continue
        base_fn = row["folder"]+"_"+row["drive"]+"_"+os.path.splitext(row["filename"])[0]
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = _read_kitti_depth(row["gt_path"])
            H, W, _  = image_np.shape; cx = W/2.0
            pred_raw  = run_da3_inference(model, image_np, device)
            pred_clip = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
            crop_mask = eigen_crop_mask(gt_depth.shape)
            valid_gt  = (gt_depth > MIN_DEPTH) & (gt_depth < MAX_DEPTH)

            # 1. Full-frame row (ALWAYS emitted)
            ff_mask = valid_gt & crop_mask
            if np.any(ff_mask):
                m = compute_depth_errors(gt_depth, pred_clip, ff_mask)
                _write_row(base_fn, idx, "fullframe", 0, "fullframe", 1.0,
                           "fullframe", "fullframe", 0.0, m,
                           float(gt_depth[ff_mask].mean()),
                           float(gt_depth[ff_mask].min()),
                           row["rgb_path"], row["gt_path"])

            # 2. Per-mask rows (only when MASK_ROOT configured)
            if MASK_ROOT:
                mask_dir  = get_mask_dir(row["rgb_path"], MASK_ROOT, KITTI_ROOT)
                instances = load_masks_for_frame(mask_dir)
                for inst in instances:
                    if inst["score"] < MIN_MASK_SCORE: continue
                    seg = inst["mask"]
                    if seg.shape != (H, W): continue
                    fm = seg & valid_gt & crop_mask
                    if not np.any(fm): continue
                    m = compute_depth_errors(gt_depth, pred_clip, fm)
                    _, xc = np.where(seg)
                    angle = math.degrees(math.atan(
                        (float(np.mean(xc))-cx)/FOCAL_LENGTH_X)) if len(xc) else 0.0
                    _write_row(base_fn, idx, "mask",
                               inst["local_idx"], str(inst.get("uid","")),
                               inst["score"], inst["super_cat"], inst["sub_cat"],
                               angle, m,
                               float(gt_depth[fm].mean()), float(gt_depth[fm].min()),
                               row["rgb_path"], row["gt_path"])
        except Exception as e:
            log.error("Error %s: %s", base_fn, e, exc_info=True)
    log.info("Evaluation complete -> %s", OUTPUT_CSV_PATH)


def _write_row(base_fn, idx, row_type, mask_id, mask_uid, score,
               super_cat, sub_cat, angle, metrics, la, lm, rp, gp):
    r = metrics
    pd.DataFrame([{
        "img_id": idx, "file_name": base_fn, "row_type": row_type,
        "mask_id": mask_id, "mask_uid": mask_uid, "mask_score": score,
        "super_cat": super_cat, "sub_cat": sub_cat, "horizontal_angle": angle,
        "rmse": r["rmse"], "abs_rel": r["abs_rel"], "sq_rel": r["sq_rel"],
        "rmse_log": r["rmse_log"], "delta1": r["delta1"],
        "delta2": r["delta2"], "delta3": r["delta3"],
        "avg_pred_depth": r["avg_pred_depth"],
        "sum_sq_err": r["sum_sq_err"], "n_pixels": r["n_pixels"],
        "lidar_avg_depth": la, "lidar_min_depth": lm,
        "rgb_image_path": rp, "gt_depth_map_path": gp,
    }]).to_csv(OUTPUT_CSV_PATH, mode="a", header=False, index=False)


def quick_test_rmse(model, test_index, device,
                    max_samples: int = PERIODIC_TEST_SAMPLES) -> float:
    """Pixel-weighted RMSE on a deterministic subsample -- for per-epoch test curve."""
    idx_sub = test_index
    if max_samples > 0 and len(test_index) > max_samples:
        rng     = np.random.RandomState(SPLIT_SEED)
        idx_sub = test_index.iloc[rng.choice(len(test_index), max_samples, replace=False)]
    model.eval(); total_sse = 0.0; total_npx = 0
    for _, row in tqdm(idx_sub.iterrows(), total=len(idx_sub),
                       desc="  quick test RMSE", leave=False):
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = _read_kitti_depth(row["gt_path"])
            pred_raw = run_da3_inference(model, image_np, device)
            pred_c   = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
            mask     = (gt_depth > MIN_DEPTH) & (gt_depth < MAX_DEPTH) & eigen_crop_mask(gt_depth.shape)
            if not np.any(mask): continue
            err = (gt_depth[mask] - pred_c[mask])**2
            total_sse += float(err.sum()); total_npx += int(mask.sum())
        except Exception: pass
    return float(np.sqrt(total_sse / max(total_npx, 1)))
# ─────────────────────────────────────────────────────────────────────────────


## 13 · Analysis & Visualisation  (12 plots)

| # | Plot | Description |
|---|------|-------------|
| 00 | Summary table | Metric overview across all scale methods |
| 01 | GT vs Predicted depth | Scatter with regression for all 4 variants |
| 02 | GT depth vs RMSE | Error-distance relationship |
| 03 | Angle vs RMSE | Lateral-angle error analysis |
| 04 | Scale comparison bars | RMSE / AbsRel / δ<1.25 bar chart |
| 05 | Model vs LiDAR by category | Per-category scatter |
| 06 | RMSE distributions | Histograms with mean/median lines |
| 07 | Scale factor distributions | Histograms centred at 1.0 |
| 08 | 3D error surfaces | Polynomial regression over depth × angle |
| 09 | Depth-bin analysis | RMSE and bias by 10 m bins |
| 10 | RMSE by category | Grouped bar chart per object class |
| 11 | Scale factor vs depth | LOWESS regression curves |

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# Analysis & visualisation -- UNSCALED METRICS ONLY.
#
# Plots:
#  00  Summary table image (full-frame + super-cat + sub-cat, pixel-weighted)
#  01  Training curves (train_loss, val_loss, val_rmse vs epoch)
#  02  Val RMSE + test RMSE vs epoch (from periodic checkpoints)
#  03  GT depth vs predicted depth scatter (full-frame)
#  04  GT depth vs RMSE scatter (full-frame)
#  05  Horizontal angle vs RMSE (per-mask)
#  06  RMSE distribution histogram (full-frame)
#  07  Depth-bin RMSE & bias (full-frame)
#  08  Full-frame vs super-category RMSE bar (pixel-weighted)
#  09  Sub-category PW-RMSE heatmap
#  10  3-D error surface (depth x angle x RMSE, per-mask)
# ─────────────────────────────────────────────────────────────────────────────

def _save_fig(fig, name, dpi=150):
    os.makedirs(PLOT_DIR, exist_ok=True)
    fig.savefig(os.path.join(PLOT_DIR, name), dpi=dpi, bbox_inches="tight")
    plt.close(fig); log.info("  Saved: %s", name)


def load_results() -> pd.DataFrame:
    df = pd.read_csv(OUTPUT_CSV_PATH)
    return df.dropna(subset=["rmse","n_pixels","sum_sq_err"])

def _ff(df): return df[df["row_type"] == "fullframe"]
def _mask(df): return df[df["row_type"] == "mask"]


# 00 -- Summary table image
def plot_summary_table_image(df):
    ff = _ff(df); dm = _mask(df); rows = []
    if len(ff):
        rows.append(["FULL-FRAME (Eigen crop)", "--",
                     f"{pw_rmse(ff):.4f}", f"{ff['abs_rel'].mean():.4f}",
                     f"{ff['delta1'].mean():.4f}", f"{len(ff):,}", f"{ff['n_pixels'].sum():,}"])
    if len(dm):
        for cat in sorted(dm["super_cat"].dropna().unique()):
            sub = dm[dm["super_cat"]==cat]
            rows.append([f"  {cat}", "super-cat",
                         f"{pw_rmse(sub):.4f}", f"{sub['abs_rel'].mean():.4f}",
                         f"{sub['delta1'].mean():.4f}", f"{len(sub):,}", f"{sub['n_pixels'].sum():,}"])
            for scat in sorted(sub["sub_cat"].dropna().unique()):
                ss = sub[sub["sub_cat"]==scat]
                rows.append([f"    {scat}  [{cat}]", "sub-cat",
                             f"{pw_rmse(ss):.4f}", f"{ss['abs_rel'].mean():.4f}",
                             f"{ss['delta1'].mean():.4f}", f"{len(ss):,}", f"{ss['n_pixels'].sum():,}"])
    fig_h = max(3, 0.45*len(rows)+1.5)
    fig, ax = plt.subplots(figsize=(16, fig_h)); ax.axis("off")
    hdrs = ["Row", "Type", "PW-RMSE (m)", "Abs Rel", "delta<1.25", "N rows", "N pixels"]
    tbl  = ax.table(cellText=rows, colLabels=hdrs, cellLoc="center", loc="center", bbox=[0,0,1,1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.6)
    for j in range(len(hdrs)):
        tbl[0,j].set_facecolor("#2c3e50"); tbl[0,j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows)+1):
        bg = "white" if i%2==0 else "#f0f4f8"
        for j in range(len(hdrs)): tbl[i,j].set_facecolor(bg)
    fig.suptitle(f"DA3METRIC-LARGE (Fine-tuned, {LOSS_NAME}) -- Evaluation Summary",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout(); _save_fig(fig, "00_summary_table.png", dpi=180)


# 01 -- Training curves
def plot_training_curves(history: List[Dict]) -> None:
    if not history: log.warning("History empty -- skip training curves."); return
    eps = [h["epoch"] for h in history]
    tl  = [h["train_loss"] for h in history]
    vl  = [h["val_loss"]   for h in history]
    vr  = [h["val_rmse"]   for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(eps, tl, "o-", color="#e74c3c", linewidth=2, markersize=4)
    axes[0].set_title("Train Loss vs Epoch"); axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss"); axes[0].grid(alpha=0.3)
    axes[1].plot(eps, vl, "o-", color="#2980b9", linewidth=2, markersize=4)
    axes[1].set_title("Val Loss vs Epoch"); axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss"); axes[1].grid(alpha=0.3)
    best_e = eps[int(np.argmin(vr))]; best_v = min(vr)
    axes[2].plot(eps, vr, "o-", color="#27ae60", linewidth=2, markersize=4)
    axes[2].axvline(best_e, color="red", linestyle="--", linewidth=1.5,
                    label=f"Best ep {best_e}: {best_v:.4f} m")
    axes[2].set_title("Val RMSE (pixel-weighted) vs Epoch"); axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("RMSE (m)"); axes[2].grid(alpha=0.3); axes[2].legend()
    fig.suptitle(f"Training Curves -- {LOSS_NAME}", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "01_training_curves.png")


# 02 -- Val + test RMSE vs epoch
def plot_epoch_rmse(history: List[Dict]) -> None:
    if not history: return
    fig, ax = plt.subplots(figsize=(12, 5))
    eps = [h["epoch"] for h in history]; vr = [h["val_rmse"] for h in history]
    ax.plot(eps, vr, "o-", color="#27ae60", linewidth=2, markersize=5,
            label="Val RMSE (pixel-weighted)")
    if os.path.exists(PERIODIC_CSV_PATH):
        pr = pd.read_csv(PERIODIC_CSV_PATH).dropna(subset=["test_rmse"])
        if len(pr):
            ax.plot(pr["epoch"], pr["test_rmse"], "s--", color="#e67e22",
                    linewidth=2, markersize=6, label="Test RMSE (periodic ckpt, subset)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE (m)")
    ax.set_title(f"Val RMSE & Test RMSE vs Epoch -- {LOSS_NAME}", fontsize=13)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); _save_fig(fig, "02_val_test_rmse_vs_epoch.png")


# 03 -- GT vs pred scatter (full-frame)
def plot_gt_vs_pred(df):
    ff = _ff(df)
    if len(ff) == 0: return
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.regplot(ax=ax, data=ff, x="lidar_avg_depth", y="avg_pred_depth",
                scatter_kws={"s": 20, "alpha": 0.3, "color": "#e74c3c"},
                line_kws={"color": "blue", "linewidth": 2}, lowess=True)
    ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
    ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
    ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
    ax.set_title(f"GT vs Predicted Depth (Unscaled) -- {LOSS_NAME}"); ax.legend()
    plt.tight_layout(); _save_fig(fig, "03_gt_vs_pred.png")


# 04 -- GT vs RMSE scatter (full-frame)
def plot_gt_vs_rmse(df):
    ff = _ff(df)
    if len(ff) == 0: return
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.regplot(ax=ax, data=ff, x="lidar_avg_depth", y="rmse",
                scatter_kws={"s": 20, "alpha": 0.3, "color": "#e74c3c"},
                line_kws={"color": "blue", "linewidth": 2}, lowess=True)
    ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("RMSE (m)")
    ax.set_title(f"GT Depth vs RMSE (Full-frame) -- {LOSS_NAME}")
    ax.set_xlim(0, MAX_DEPTH); ax.grid(alpha=0.3)
    plt.tight_layout(); _save_fig(fig, "04_gt_vs_rmse.png")


# 05 -- Angle vs RMSE (per-mask)
def plot_angle_vs_rmse(df):
    dm = _mask(df)
    if len(dm) == 0 or "horizontal_angle" not in dm.columns: return
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.regplot(ax=ax, data=dm, x="horizontal_angle", y="rmse",
                scatter_kws={"s": 15, "alpha": 0.25, "color": "#8e44ad"},
                line_kws={"color": "red", "linewidth": 2}, lowess=True)
    ax.set_xlabel("Horizontal Angle (deg)"); ax.set_ylabel("RMSE (m)")
    ax.set_title(f"Angle vs RMSE (per-mask, unscaled) -- {LOSS_NAME}"); ax.grid(alpha=0.3)
    plt.tight_layout(); _save_fig(fig, "05_angle_vs_rmse.png")


# 06 -- RMSE distribution (full-frame)
def plot_rmse_distribution(df):
    ff = _ff(df)
    if len(ff) == 0: return
    vals = ff["rmse"].dropna(); pw = pw_rmse(ff)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(vals, bins=60, color="#e74c3c", alpha=0.75, edgecolor="white")
    ax.axvline(vals.mean(),   color="black", linestyle="--", linewidth=1.8,
               label=f"Mean   = {vals.mean():.3f} m")
    ax.axvline(vals.median(), color="grey",  linestyle=":",  linewidth=1.8,
               label=f"Median = {vals.median():.3f} m")
    ax.axvline(pw, color="blue", linestyle="-.", linewidth=1.8,
               label=f"PW-RMSE = {pw:.3f} m")
    ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Count"); ax.legend()
    ax.set_title(f"Full-frame RMSE Distribution -- {LOSS_NAME}")
    plt.tight_layout(); _save_fig(fig, "06_rmse_distribution.png")


# 07 -- Depth-bin RMSE & bias (full-frame)
def plot_depth_bin_analysis(df):
    ff = _ff(df)
    if len(ff) == 0: return
    bins   = np.arange(0, MAX_DEPTH+10, 10)
    labels = [f"{int(b)}-{int(b+10)}m" for b in bins[:-1]]
    df2    = ff.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_avg_depth"], bins=bins, labels=labels)
    bin_pw = {bl: pw_rmse(df2[df2["depth_bin"]==bl]) for bl in labels}
    bias   = df2.groupby("depth_bin", observed=True).apply(
                 lambda g: g["avg_pred_depth"].mean() - g["lidar_avg_depth"].mean())
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].plot(labels, [bin_pw[b] for b in labels], "o-", color="#e74c3c",
                 linewidth=2, label="PW-RMSE")
    axes[0].set_xlabel("Depth Bin"); axes[0].set_ylabel("PW-RMSE (m)")
    axes[0].set_title("RMSE by Depth Bin (Full-frame, unscaled)")
    axes[0].tick_params(axis="x", rotation=45); axes[0].grid(alpha=0.3); axes[0].legend()
    axes[1].plot(bias.index, bias.values, "o-", color="#2980b9", linewidth=2)
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1.5)
    axes[1].set_xlabel("Depth Bin"); axes[1].set_ylabel("Pred Bias (m)")
    axes[1].set_title("Depth Bias (Pred - LiDAR)")
    axes[1].tick_params(axis="x", rotation=45); axes[1].grid(alpha=0.3)
    fig.suptitle(f"Depth-Bin Analysis -- {LOSS_NAME}", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "07_depth_bin_analysis.png")


# 08 -- Full-frame vs category RMSE bar (pixel-weighted)
def plot_rmse_by_category(df):
    ff = _ff(df); dm = _mask(df)
    cats   = sorted(dm["super_cat"].dropna().unique()) if len(dm) else []
    labels = ["Full-frame"] + list(cats)
    values = ([pw_rmse(ff)] if len(ff) else [float("nan")]) +              [pw_rmse(dm[dm["super_cat"]==c]) for c in cats]
    npix   = ([ff["n_pixels"].sum()] if len(ff) else [0]) +              [int(dm[dm["super_cat"]==c]["n_pixels"].sum()) for c in cats]
    colors = ["#2c3e50"] + sns.color_palette("Set2", max(len(cats),1)).as_hex()
    fig, ax = plt.subplots(figsize=(max(8, 2.5*len(labels)), 6))
    bars = ax.bar(labels, values, color=colors[:len(labels)], edgecolor="white", width=0.6)
    for bar, v, np_ in zip(bars, values, npix):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{v:.3f} m  ({np_/1e6:.1f}M px)", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Pixel-weighted RMSE (m)")
    ax.set_title(f"Full-frame vs Category RMSE (pixel-weighted) -- {LOSS_NAME}", fontsize=13)
    ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save_fig(fig, "08_rmse_by_category_pw.png")


# 09 -- Sub-category heatmap
def plot_subcat_heatmap(df):
    dm = _mask(df)
    if len(dm) == 0: return
    super_cats = sorted(dm["super_cat"].dropna().unique())
    sub_cats   = sorted(dm["sub_cat"].dropna().unique())
    if not super_cats or not sub_cats: return
    grid = np.full((len(sub_cats), len(super_cats)), np.nan)
    for ci, sc in enumerate(super_cats):
        for ri, ss in enumerate(sub_cats):
            sub = dm[(dm["super_cat"]==sc)&(dm["sub_cat"]==ss)]
            if len(sub): grid[ri, ci] = pw_rmse(sub)
    fig, ax = plt.subplots(figsize=(max(8, 2*len(super_cats)), max(5, 0.5*len(sub_cats))))
    im = ax.imshow(grid, aspect="auto", cmap="RdYlGn_r")
    plt.colorbar(im, ax=ax, label="PW-RMSE (m)")
    ax.set_xticks(range(len(super_cats))); ax.set_xticklabels(super_cats, rotation=30)
    ax.set_yticks(range(len(sub_cats)));   ax.set_yticklabels(sub_cats, fontsize=8)
    for (r,c), v in np.ndenumerate(grid):
        if not np.isnan(v):
            ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(f"Sub-category PW-RMSE Heatmap -- {LOSS_NAME}", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "09_subcat_heatmap.png")


# 10 -- 3-D error surface (per-mask)
def plot_3d_error_surface(df):
    dm = _mask(df)
    if len(dm) < 10 or "horizontal_angle" not in dm.columns: return
    dc = dm.dropna(subset=["lidar_avg_depth","horizontal_angle","rmse"])
    if len(dc) < 10: return
    fig = plt.figure(figsize=(10, 7)); ax = fig.add_subplot(111, projection="3d")
    ax.scatter(dc["lidar_avg_depth"], dc["horizontal_angle"], dc["rmse"],
               c="#e74c3c", alpha=0.25, s=6)
    X  = dc[["lidar_avg_depth","horizontal_angle"]].values; z = dc["rmse"].values
    pipe = make_pipeline(PolynomialFeatures(2), LinearRegression()); pipe.fit(X, z)
    xs, ys = np.meshgrid(np.linspace(dc["lidar_avg_depth"].min(),dc["lidar_avg_depth"].max(),30),
                         np.linspace(dc["horizontal_angle"].min(),dc["horizontal_angle"].max(),30))
    zs = pipe.predict(np.c_[xs.ravel(), ys.ravel()]).reshape(xs.shape)
    ax.plot_surface(xs, ys, zs, color="#2980b9", alpha=0.25)
    ax.set_xlabel("GT Depth (m)"); ax.set_ylabel("H. Angle (deg)"); ax.set_zlabel("RMSE (m)")
    ax.set_title(f"3-D Error Surface (per-mask, unscaled) -- {LOSS_NAME}", fontsize=12)
    plt.tight_layout(); _save_fig(fig, "10_3d_error_surface.png")


def generate_all_plots(df, history=None):
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    log.info("Generating plots -> %s", PLOT_DIR)
    h = history if history is not None else load_history()
    plot_summary_table_image(df)
    plot_training_curves(h)
    plot_epoch_rmse(h)
    plot_gt_vs_pred(df)
    plot_gt_vs_rmse(df)
    plot_angle_vs_rmse(df)
    plot_rmse_distribution(df)
    plot_depth_bin_analysis(df)
    plot_rmse_by_category(df)
    plot_subcat_heatmap(df)
    plot_3d_error_surface(df)
    log.info("All plots saved -> %s", PLOT_DIR)
# ─────────────────────────────────────────────────────────────────────────────


## 14 · Terminal Summary Printer

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
def print_summary(df: pd.DataFrame) -> None:
    W = 125; ff = df[df["row_type"]=="fullframe"]; dm = df[df["row_type"]=="mask"]
    print("=" * W)
    print(f"  Fine-tuned DA3METRIC-LARGE ({LOSS_NAME}) -- Evaluation Summary")
    print(f"  Frames: {df['img_id'].nunique():,}  |  "
          f"Full-frame rows: {len(ff):,}  |  Mask rows: {len(dm):,}")
    print("="*W)
    hdr = (f"  {'Row':<38}{'Type':<12}"
           f"{'PW-RMSE':>9}{'AbsRel':>9}{'SqRel':>9}"
           f"{'RMSElog':>9}{'d1':>8}{'d2':>8}{'d3':>8}{'N-px(M)':>9}")
    print(hdr); print("-"*W)

    def _row(label, rtype, sub):
        if len(sub) == 0: return
        print(f"  {label:<38}{rtype:<12}"
              f"{pw_rmse(sub):>9.4f}{sub['abs_rel'].mean():>9.4f}"
              f"{sub['sq_rel'].mean():>9.4f}{sub['rmse_log'].mean():>9.4f}"
              f"{sub['delta1'].mean():>8.4f}{sub['delta2'].mean():>8.4f}"
              f"{sub['delta3'].mean():>8.4f}{sub['n_pixels'].sum()/1e6:>9.2f}")

    _row("Full-frame (Eigen crop)", "fullframe", ff)
    if len(dm):
        print("-"*W); print("  Super-category breakdown (pixel-weighted)"); print("-"*W)
        for cat in sorted(dm["super_cat"].dropna().unique()):
            _row(f"  {cat}", "super-cat", dm[dm["super_cat"]==cat])
        print("-"*W); print("  Sub-category breakdown"); print("-"*W)
        for cat in sorted(dm["super_cat"].dropna().unique()):
            for scat in sorted(dm[dm["super_cat"]==cat]["sub_cat"].dropna().unique()):
                ss = dm[(dm["super_cat"]==cat)&(dm["sub_cat"]==scat)]
                _row(f"    {scat}  [{cat}]", "sub-cat", ss)
    print("="*W); print()
# ─────────────────────────────────────────────────────────────────────────────


## 15 · Main Orchestration Function

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_periodic_checkpoints(device: torch.device) -> None:
    """
    Load each checkpoint_epoch_NNNN.pth, run quick_test_rmse, and record
    (epoch, val_rmse, test_rmse) in PERIODIC_CSV_PATH.
    Skips epochs already present in the CSV (resume-safe).
    """
    periodic = list_periodic_checkpoints(CHECKPOINT_DIR)
    if not periodic:
        log.warning("No periodic checkpoints in %s -- skipping.", CHECKPOINT_DIR); return
    history      = load_history()
    hist_by_ep   = {h["epoch"]: h for h in history}
    done_epochs: set = set()
    if os.path.exists(PERIODIC_CSV_PATH):
        done_epochs = set(pd.read_csv(PERIODIC_CSV_PATH)["epoch"].tolist())
    test_index = index_kitti_test(TEST_ROOT)
    rows = []
    for ckpt_path in periodic:
        epoch = int(Path(ckpt_path).stem.replace("checkpoint_epoch_", ""))
        if epoch in done_epochs:
            log.info("  Epoch %d already evaluated -- skipping", epoch); continue
        log.info("  Evaluating epoch %d ...", epoch)
        model  = load_da3_model(device, checkpoint_path=ckpt_path)
        test_r = quick_test_rmse(model, test_index, device)
        val_r  = hist_by_ep.get(epoch, {}).get("val_rmse", float("nan"))
        tl     = hist_by_ep.get(epoch, {}).get("train_loss", float("nan"))
        vl     = hist_by_ep.get(epoch, {}).get("val_loss", float("nan"))
        rows.append({"epoch": epoch, "test_rmse": test_r, "val_rmse": val_r,
                     "train_loss": tl, "val_loss": vl})
        log.info("    epoch %d  test_rmse=%.4f  val_rmse=%.4f", epoch, test_r, val_r)
        del model; torch.cuda.empty_cache()
    if rows:
        new_df = pd.DataFrame(rows)
        if os.path.exists(PERIODIC_CSV_PATH):
            new_df = pd.concat([pd.read_csv(PERIODIC_CSV_PATH), new_df]).drop_duplicates("epoch")
        new_df.sort_values("epoch").to_csv(PERIODIC_CSV_PATH, index=False)
        log.info("Periodic test metrics -> %s", PERIODIC_CSV_PATH)


def main(skip_training=False, skip_inference=False, skip_plots=False,
         skip_periodic=False, eval_start_idx=0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info("Device: %s", device)
    if device.type == "cuda":
        log.info("GPU: %s  VRAM: %.1f GB", torch.cuda.get_device_name(0),
                 torch.cuda.get_device_properties(0).total_memory/1e9)

    # Step 1: Training
    if not skip_training:
        log.info("STEP 1/4: Fine-tuning")
        best_ckpt = run_training(device)
    else:
        best_ckpt = os.path.join(CHECKPOINT_DIR, "best_model.pth")
        if not os.path.exists(best_ckpt): best_ckpt = find_resume_checkpoint(CHECKPOINT_DIR)
        log.info("STEP 1/4: Skipped (ckpt: %s)", best_ckpt)

    # Step 2: Evaluate best checkpoint on test set
    if not skip_inference:
        log.info("STEP 2/4: Evaluating best model on KITTI test split")
        model      = load_da3_model(device, checkpoint_path=best_ckpt)
        test_index = index_kitti_test(TEST_ROOT)
        run_evaluation(model, test_index, device, start_index=eval_start_idx)
    else:
        log.info("STEP 2/4: Skipped (CSV: %s)", OUTPUT_CSV_PATH)

    # Step 3: Per-epoch test RMSE via periodic checkpoints
    if not skip_periodic:
        log.info("STEP 3/4: Evaluating periodic checkpoints (test_rmse per epoch)")
        evaluate_periodic_checkpoints(device)
    else:
        log.info("STEP 3/4: Skipped")

    # Step 4: Plots & summary
    log.info("STEP 4/4: Analysis, plots & summary")
    if os.path.exists(OUTPUT_CSV_PATH):
        df = load_results(); history = load_history()
        log.info("Loaded %d rows from CSV", len(df))
        print_summary(df)
        if not skip_plots: generate_all_plots(df, history)
    else:
        log.warning("CSV not found at %s", OUTPUT_CSV_PATH)
# ─────────────────────────────────────────────────────────────────────────────


## 16 · ▶ Run the Pipeline

Set `LOSS_NAME` in **Section 1** to choose one of the seven recipes
(`A1`, `A2`, `A3`, `A4`, `B1`, `B2`, `C`). Outputs are written to
`./da3_kitti_finetune_final_outputs/loss_<LOSS_NAME>/` so each recipe keeps
its own checkpoints, eval CSV, and plots.

Set the execution flags below to control which steps run, then execute the
cell. Training resumes automatically from the latest checkpoint in
`CHECKPOINT_DIR` if one exists.


In [18]:
for d in [OUTPUT_BASE, CHECKPOINT_DIR, EVAL_OUTPUT_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)
log.info("Output root: %s", os.path.abspath(OUTPUT_BASE))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Device: %s", DEVICE)
if DEVICE.type == "cuda":
    log.info("GPU: %s  |  VRAM: %.1f GB",
             torch.cuda.get_device_name(0),
             torch.cuda.get_device_properties(0).total_memory/1e9)

main(skip_training=SKIP_TRAINING, skip_inference=SKIP_INFERENCE,
     skip_plots=SKIP_PLOTS, skip_periodic=SKIP_PERIODIC,
     eval_start_idx=EVAL_START_IDX)


09:25:34  INFO      Output root: /home/arka/Downloads/MMDE/outputs/fully_supervised_FT_output/da3_kitti_finetune_final_outputs_complete_v4_FULL_ARCH/loss_A1
09:25:34  INFO      Device: cuda
09:25:34  INFO      GPU: Tesla V100-PCIE-32GB  |  VRAM: 34.1 GB
09:25:34  INFO      Device: cuda
09:25:34  INFO      GPU: Tesla V100-PCIE-32GB  VRAM: 34.1 GB
09:25:34  INFO      STEP 1/4: Fine-tuning
09:25:34  INFO      Recipe: A1  optim=adamw  lr=2.0e-07  mae_floor=0.500
09:25:34  INFO      Building 90/10 train-val split (val_frac=0.10, seed=42)
09:25:36  INFO      [KITTI-train]  23158 samples  (target 1246x378)
09:25:37  INFO      [KITTI-train]  23158 samples  (target 1246x378)
09:25:37  INFO        train=20842  val=2316  (total 23158)
09:25:37  INFO      Train: 5210 batches | Val: 2316 batches
09:25:38  INFO      Loading depth-anything/DA3METRIC-LARGE ...


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


09:25:38  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
09:25:38  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/depth-anything/DA3METRIC-LARGE/4010e39f3634a45bc60553321fb49fb760bd594e/config.json "HTTP/1.1 200 OK"


[INFO ] using MLP layer as FFN


09:25:41  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/model.safetensors "HTTP/1.1 302 Found"
09:25:41  INFO        Parameters: 334.2M total, 334.2M trainable
09:25:41  INFO      Architecture: Full-model fine-tuning
09:25:41  INFO      Trainable parameters: 334.17M / 334.17M
09:25:41  INFO      Loss recipe A1 : 1*mae_canonical + 1*silog + 0.1*grad
09:25:41  INFO        MAECanonical depth_floor=0.500 m  (1/pred <= 2.0)
09:25:41  INFO      Optimizer: AdamW  lr=2.0e-07  wd=1.0e-04
09:25:41  INFO      Scheduler: cosine_warmup  warm_steps=2605  main_steps=127645  eta_min=2.0e-09
09:25:41  INFO      Fresh training (no checkpoint found)
09:25:41  INFO      ======================================================================
09:25:41  INFO      Loss=A1  Arch=full  Aug=base  Optim=adamw
09:25:41  INFO      Epochs: 1->50  BS=4x2=8  LR=2.0e-07  MAE_floor=0.500 m
09:25:41  INFO      AMP=True  Clip=1.0  Patience=30  SaveEvery=1  Device=cuda
09:25

Val Ep1:   0%|          | 0/2316 [00:38<?, ?it/s]

... (197 outputs left out)

  quick test RMSE:   0%|          | 0/652 [00:00<?, ?it/s]

22:49:30  INFO          epoch 32  test_rmse=3.0548  val_rmse=2.6139
22:49:30  INFO      Periodic test metrics -> /home/arka/Downloads/MMDE/outputs/fully_supervised_FT_output/da3_kitti_finetune_final_outputs_complete_v4_FULL_ARCH/loss_A1/eval_results/periodic_test_metrics.csv
22:49:30  INFO      STEP 4/4: Analysis, plots & summary
22:49:30  INFO      Loaded 13348 rows from CSV
22:49:31  INFO      Generating plots -> /home/arka/Downloads/MMDE/outputs/fully_supervised_FT_output/da3_kitti_finetune_final_outputs_complete_v4_FULL_ARCH/loss_A1/eval_results/plots


  Fine-tuned DA3METRIC-LARGE (A1) -- Evaluation Summary
  Frames: 652  |  Full-frame rows: 652  |  Mask rows: 12,696
  Row                                   Type          PW-RMSE   AbsRel    SqRel  RMSElog      d1      d2      d3  N-px(M)
-----------------------------------------------------------------------------------------------------------------------------
  Full-frame (Eigen crop)               fullframe      2.2459   0.0509   0.1632   0.0760  0.9813  0.9971  0.9990    46.21
-----------------------------------------------------------------------------------------------------------------------------
  Super-category breakdown (pixel-weighted)
-----------------------------------------------------------------------------------------------------------------------------
    animal                              super-cat      0.6055   0.0780   0.0562   0.0876  0.9868  1.0000  1.0000     0.00
    cyclist                             super-cat      1.6389   0.0608   0.1898   0.0748  0.979

22:49:33  INFO        Saved: 00_summary_table.png
22:49:34  INFO        Saved: 01_training_curves.png
22:49:34  INFO        Saved: 02_val_test_rmse_vs_epoch.png
22:49:34  INFO        Saved: 03_gt_vs_pred.png
22:49:35  INFO        Saved: 04_gt_vs_rmse.png
22:49:41  INFO        Saved: 05_angle_vs_rmse.png
22:49:42  INFO        Saved: 06_rmse_distribution.png
22:49:42  INFO        Saved: 07_depth_bin_analysis.png
22:49:43  INFO        Saved: 08_rmse_by_category_pw.png
22:49:44  INFO        Saved: 09_subcat_heatmap.png
22:49:45  INFO        Saved: 10_3d_error_surface.png
22:49:45  INFO      All plots saved -> /home/arka/Downloads/MMDE/outputs/fully_supervised_FT_output/da3_kitti_finetune_final_outputs_complete_v4_FULL_ARCH/loss_A1/eval_results/plots


## 17 · Standalone Analysis  *(run independently if CSV exists)*

Use this cell to regenerate plots or print the summary from an existing results CSV
**without re-running inference**.

In [19]:
# ── Standalone: load existing CSV and regenerate everything ──────────────────
if os.path.exists(OUTPUT_CSV_PATH):
    df = load_results()
    log.info("Loaded %d rows from CSV", len(df))

    # Terminal summary
    print_summary(df)

    # Regenerate all 12 plots
    generate_all_plots(df)
    log.info("Plots saved to: %s", PLOT_DIR)
else:
    log.warning("CSV not found at %s — run inference first.", OUTPUT_CSV_PATH)

22:49:46  INFO      Loaded 13348 rows from CSV
22:49:46  INFO      Generating plots -> /home/arka/Downloads/MMDE/outputs/fully_supervised_FT_output/da3_kitti_finetune_final_outputs_complete_v4_FULL_ARCH/loss_A1/eval_results/plots


  Fine-tuned DA3METRIC-LARGE (A1) -- Evaluation Summary
  Frames: 652  |  Full-frame rows: 652  |  Mask rows: 12,696
  Row                                   Type          PW-RMSE   AbsRel    SqRel  RMSElog      d1      d2      d3  N-px(M)
-----------------------------------------------------------------------------------------------------------------------------
  Full-frame (Eigen crop)               fullframe      2.2459   0.0509   0.1632   0.0760  0.9813  0.9971  0.9990    46.21
-----------------------------------------------------------------------------------------------------------------------------
  Super-category breakdown (pixel-weighted)
-----------------------------------------------------------------------------------------------------------------------------
    animal                              super-cat      0.6055   0.0780   0.0562   0.0876  0.9868  1.0000  1.0000     0.00
    cyclist                             super-cat      1.6389   0.0608   0.1898   0.0748  0.979

22:49:48  INFO        Saved: 00_summary_table.png
22:49:49  INFO        Saved: 01_training_curves.png
22:49:49  INFO        Saved: 02_val_test_rmse_vs_epoch.png
22:49:49  INFO        Saved: 03_gt_vs_pred.png
22:49:49  INFO        Saved: 04_gt_vs_rmse.png
22:49:56  INFO        Saved: 05_angle_vs_rmse.png
22:49:56  INFO        Saved: 06_rmse_distribution.png
22:49:57  INFO        Saved: 07_depth_bin_analysis.png
22:49:57  INFO        Saved: 08_rmse_by_category_pw.png
22:49:59  INFO        Saved: 09_subcat_heatmap.png
22:49:59  INFO        Saved: 10_3d_error_surface.png
22:49:59  INFO      All plots saved -> /home/arka/Downloads/MMDE/outputs/fully_supervised_FT_output/da3_kitti_finetune_final_outputs_complete_v4_FULL_ARCH/loss_A1/eval_results/plots
22:49:59  INFO      Plots saved to: /home/arka/Downloads/MMDE/outputs/fully_supervised_FT_output/da3_kitti_finetune_final_outputs_complete_v4_FULL_ARCH/loss_A1/eval_results/plots
